In [ ]:

import numpy as np
import pandas as pd
import json, subprocess, time, pathlib
from scipy import stats
from datetime import datetime, timezone

SHARED            = pathlib.Path("/workspace/shared")
CANARY_LOG        = SHARED / "canary_log.jsonl"
METRICS_CSV       = SHARED / "minicluster/live_metrics.csv"
ROLLBACK_LOG      = SHARED / "rollback_log.jsonl"
SERVICES          = ["payments", "auth", "checkout", "fraud"]
P_THRESHOLD       = 0.05
LATENCY_THRESH_MS = 300
ERROR_THRESH      = 0.10
LAT_COL           = "latency_p95_ms"

def load_metrics(service, last_n_rows=200):
    df = pd.read_csv(METRICS_CSV)
    df.columns = df.columns.str.strip().str.lower().str.replace(" ","_")
    df["ts"] = pd.to_datetime(df["timestamp"], errors="coerce")
    df = df.dropna(subset=["ts"]).sort_values("ts")
    return df[df["service"].str.lower()==service.lower()].tail(last_n_rows).reset_index(drop=True)

def canary_score(baseline, canary):
    if len(baseline)<5 or len(canary)<3:
        return {"score":50,"verdict":"INSUFFICIENT_DATA","mw_pvalue":1.0,"mean_delta":0.0}
    _, p       = stats.mannwhitneyu(baseline, canary, alternative="two-sided")
    mean_delta = (canary.mean()-baseline.mean())/(baseline.mean()+1e-9)
    score      = max(0, 100 - max(0,(1-p/P_THRESHOLD))*40 - min(40,max(0,mean_delta*100)))
    verdict    = "PASS" if score>=70 else ("WARN" if score>=45 else "FAIL")
    return {"score":round(score,1),"verdict":verdict,"mw_pvalue":round(p,4),"mean_delta":round(mean_delta,4)}

def hard_gate_check(df):
    issues=[]
    lat = df[LAT_COL].mean() if LAT_COL in df.columns else 0
    err = df["error_rate"].mean() if "error_rate" in df.columns else 0
    if lat>LATENCY_THRESH_MS: issues.append(f"P95 {lat:.0f}ms>{LATENCY_THRESH_MS}ms")
    if err>ERROR_THRESH:      issues.append(f"err {err:.3f}>{ERROR_THRESH}")
    return {"pass":len(issues)==0,"issues":issues,"lat_ms":round(lat,1),"err_rate":round(err,4)}

def analyse_canary(service, verbose=True):
    df = load_metrics(service, 300)
    if df.empty: return {"service":service,"verdict":"NO_DATA","score":0}
    split = int(len(df)*0.67)
    gate  = hard_gate_check(df.iloc[split:])
    if not gate["pass"]:
        result={"service":service,"verdict":"FAIL","score":0,
                "reason":"HARD_GATE: "+"; ".join(gate["issues"]),
                "lat_ms":gate["lat_ms"],"err_rate":gate["err_rate"],
                "mw_pvalue":None,"mean_delta":None,
                "timestamp":datetime.now(timezone.utc).isoformat()}
    else:
        sr=canary_score(df.iloc[:split][LAT_COL].values.astype(float),
                        df.iloc[split:][LAT_COL].values.astype(float))
        result={"service":service,"verdict":sr["verdict"],"score":sr["score"],
                "reason":f"MW p={sr[\'mw_pvalue\']} delta={sr[\'mean_delta\'
]:+.1%}",
                "lat_ms":gate["lat_ms"],"err_rate":gate["err_rate"],
                "mw_pvalue":sr["mw_pvalue"],"mean_delta":sr["mean_delta"],
                "timestamp":datetime.now(timezone.utc).isoformat()}
    with open(CANARY_LOG,"a") as f: f.write(json.dumps(result)+"\n")
    if verbose:
        e="✅" if result["verdict"]=="PASS" else ("⚠️" if result["verdict"]=="WARN" else "❌")
        print(f"{e} [{service.upper():8s}] score={result[\'score\'
]:5.1f} | {result[\'verdict\'
]:5s} | {result[\'reason\']}")
    return result

def execute_rollback(service, reason, score):
    action={"event_type":"AUTO_ROLLBACK","service":service,"canary_score":score,
            "reason":reason,"action":"ROLLBACK_TO_PREVIOUS_VERSION",
            "timestamp":datetime.now(timezone.utc).isoformat()}
    with open(ROLLBACK_LOG,"a") as f: f.write(json.dumps(action)+"\n")
    print(f"  🔄 AUTO-ROLLBACK → {service.upper()} | score={score} | {reason}")

# STEP 1 — Baseline
print("=== STEP 1: Baseline (no faults) ===")
[analyse_canary(s) for s in SERVICES]

# STEP 2 — Inject fault
print("\n=== STEP 2: Inject 800ms fault on payments ===")
subprocess.run(["curl","-s","-X","POST","http://127.0.0.1:7001/fault/latency",
                "-H","Content-Type: application/json","-d","{\"ms\":800}"],capture_output=True)
time.sleep(10)

# STEP 3 — Post-fault analysis
print("\n=== STEP 3: Post-fault canary ===")
for svc in SERVICES:
    r = analyse_canary(svc, verbose=True)
    if r["verdict"]=="FAIL": execute_rollback(svc, r["reason"], r["score"])

# STEP 4 — Clear
print("\n🧹 Clearing all faults...")
for port in [7001,7002,7003,7004]:
    subprocess.run(["curl","-s","-X","POST",f"http://127.0.0.1:{port}/fault/clear"],capture_output=True)
print("✅ Canary complete — logs written to canary_log.jsonl + rollback_log.jsonl")
